In [5]:
import ray
import torch
from omegaconf import OmegaConf
from tensordict import TensorDict
from tensordict.tensorclass import NonTensorStack

import transfer_queue as tq

ray.init(ignore_reinit_error=True)

config = OmegaConf.create(
    {
        "controller": {"polling_mode": True},
        "backend": {
            "storage_backend": "SimpleStorage",
            "SimpleStorage": {
                "total_storage_size": 200,
                "num_data_storage_units": 2,
            },
        },
    }
)

tq.init(config)
print("TransferQueue is ready!")

/Users/yjiang/Documents/personal/workspace/projects/verl/verl-llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-11 22:37:14,124	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-08-11 22:37:20,200	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8267 


TransferQueue is ready!


In [29]:
tq.kv_put(key='sample_0',
          partition_id='train',
          fields={"input_ids": torch.tensor([1,2,3,4], dtype=torch.int32)},
          tag={"source":"wiki", "score":0.95}
          )


KVBatchMeta(keys=['sample_0'], tags=[{'source': 'wiki', 'score': 0.95}], partition_id='train', fields=['input_ids'], extra_info={})

In [4]:
keys = ["batch_0", "batch_1", "batch_2"]

fields = TensorDict(
    {
        "input_ids": torch.tensor([[10, 20], [30, 40], [50, 60]]),
        "attention_mask": torch.ones(3, 2, dtype=torch.long),
    },
    batch_size=3,
)

tags = [
    {"split": "train", "idx": 0},
    {"split": "train", "idx": 1},
    {"split": "train", "idx": 2},
]

tq.kv_batch_put(keys=keys, partition_id="train", fields=fields, tags=tags)
print(f"Stored {len(keys)} samples in one call")

Stored 3 samples in one call


(TransferQueueController pid=2591) 2026-08-11 21:51:36,176 - ERROR - transfer_queue.controller - Error updating production status for partition train: dtype mismatch: existing=torch.int32, incoming=torch.int64. All batches for the same field must have the same dtype.
(SimpleStorageUnit pid=2598) 2026-08-11 21:51:36,173 - INFO - transfer_queue.utils.perf_utils - TQ_STORAGE_UNIT_392a27b2: [Performance] Total success requests: 1, Total req/min: 0.12, Total avg process time: 0.0005s; 
(SimpleStorageUnit pid=2598) Time range: last 8.25 minutes; 
(SimpleStorageUnit pid=2598) Per-operation statistics: PUT_DATA: req_count=1, req/min=0.12, avg_time=0.000509s, max_time=0.000509s, min_time=0.000509s


In [30]:
result=tq.kv_batch_get(keys="sample_0", partition_id="train")

In [27]:
tq.kv_clear(keys=["sample_0"], partition_id="train")

(TransferQueueController pid=2591) 2026-08-11 22:13:39,418 - INFO - transfer_queue.utils.perf_utils - TQ_CONTROLLER_f65a43c1: [Performance] Total success requests: 1, Total req/min: 0.06, Total avg process time: 0.0009s; 
(TransferQueueController pid=2591) Time range: last 17.48 minutes; 
(TransferQueueController pid=2591) Per-operation statistics: KV_RETRIEVE_META: req_count=1, req/min=0.06, avg_time=0.000947s, max_time=0.000947s, min_time=0.000947s
(SimpleStorageUnit pid=2593) Per-operation statistics: CLEAR_DATA: req_count=1, req/min=0.06, avg_time=0.000376s, max_time=0.000376s, min_time=0.000376s


In [34]:
fields_td = TensorDict(
    {
        "input_ids": torch.tensor([[5, 6, 7, 8]]),
        "attention_mask": torch.ones(1, 4, dtype=torch.long),
    },
    batch_size=1,
)

tq.kv_put(
    key="sample_1",
    partition_id="train",
    fields=fields_td,
    tag={"source": "books", "score": 0.88},
)
print("Stored sample_1")

Stored sample_1


(TransferQueueController pid=2591) 2026-08-11 22:31:02,729 - INFO - transfer_queue.utils.perf_utils - TQ_CONTROLLER_f65a43c1: [Performance] Total success requests: 12, Total req/min: 0.69, Total avg process time: 0.0008s; 
(TransferQueueController pid=2591) Time range: last 17.39 minutes; 
(TransferQueueController pid=2591) Per-operation statistics: CLEAR_META: req_count=1, req/min=0.06, avg_time=0.000203s, max_time=0.000203s, min_time=0.000203s; KV_RETRIEVE_META: req_count=7, req/min=0.40, avg_time=0.001103s, max_time=0.003701s, min_time=0.000247s; NOTIFY_DATA_UPDATE: req_count=2, req/min=0.12, avg_time=0.000722s, max_time=0.001084s, min_time=0.000361s; SET_CUSTOM_META: req_count=2, req/min=0.12, avg_time=0.000025s, max_time=0.000029s, min_time=0.000021s
(TransferQueueController pid=2591) 2026-08-11 22:31:02,738 - ERROR - transfer_queue.controller - Error updating production status for partition train: dtype mismatch: existing=torch.int32, incoming=torch.int64. All batches for the sam

In [15]:
import torch
from tensordict import TensorDict

td = TensorDict({
    "agent": TensorDict({"pos": torch.randn(10, 3), "vel": torch.randn(10, 3)}, batch_size=[10]),
    "scalar": torch.randn(10)
}, batch_size=[10])


In [33]:
# Retrieve only input_ids (single field)
result = tq.kv_batch_get(keys="sample_1", partition_id="train", select_fields="input_ids")
print("Fields returned:", list(result.keys()))
assert "input_ids" in result.keys()
assert "attention_mask" not in result.keys()

# Retrieve a specific set of fields
result = tq.kv_batch_get(
    keys="sample_1",
    partition_id="train",
    select_fields=["input_ids", "attention_mask"],
)
print("Fields returned:", list(result.keys()))

Fields returned: ['input_ids']
Fields returned: ['input_ids']


In [35]:
result = tq.kv_batch_get(
    keys="sample_1",
    partition_id="train"
)
print("Fields returned:", list(result.keys()))

Fields returned: ['input_ids']


In [36]:
print(result)

TensorDict(
    fields={
        input_ids: NestedTensor(shape=torch.Size([1, j5]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([1]),
    device=None,
    is_shared=False)


In [6]:
from tensordict import TensorDict
import torch


keys = ["batch_0", "batch_1", "batch_2"]

fields = TensorDict(
    {
        "input_ids": torch.tensor([[10, 20], [30, 40], [50, 60]]),
        "attention_mask": torch.ones(3, 2, dtype=torch.long),
    },
    batch_size=3,
)

tags = [
    {"split": "train", "idx": 0},
    {"split": "train", "idx": 1},
    {"split": "train", "idx": 2},
]

tq.kv_batch_put(keys=keys, partition_id="train", fields=fields, tags=tags)
print(f"Stored {len(keys)} samples in one call")

Stored 3 samples in one call


In [15]:
print(tq.kv_batch_get(keys=["batch_0", "batch_1"], partition_id="train")[0])

TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)


In [9]:
tq.kv_batch_get(keys="batch_0", partition_id="train", select_fields="attention_mask").keys()

_StringKeys(dict_keys(['attention_mask']))